# mBERT fine-tuning for ScamShield LK

Fine-tunes `bert-base-multilingual-cased` on the same data and splits as the TF-IDF models, and
scores it the same way, so the results drop straight into the comparison table.

**Before you start**
1. In Colab: *Runtime → Change runtime type → T4 GPU*.
2. On your PC, run `python src/preprocessing.py` in `ml/` so that `ml/data/processed/dataset.csv` exists.
3. Run the cells in order. The upload cell asks for `dataset.csv`.

**Privacy:** `dataset.csv` contains your private real messages. Keep this notebook and any
Drive copy private, and delete the uploaded file when you finish (the last cell does that).

**Protocol** (same as `ml/data/DATASET_README.md`):
- train on `synth_train`, pick the best epoch on `synth_val`, never train on real messages;
- report scam vs not-scam on `real_eval` with 95% bootstrap ranges, which is the headline;
- report 3-label macro-F1 on `synth_test`;
- report the detection rate on `real_reconstructed` separately.

In [ ]:
!pip -q install transformers datasets accelerate
import torch, transformers, sklearn
print("torch", torch.__version__, "| transformers", transformers.__version__, "| sklearn", sklearn.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, switch the runtime to a GPU")

In [ ]:
# Upload ml/data/processed/dataset.csv from your PC.
from google.colab import files
uploaded = files.upload()
DATASET = next(name for name in uploaded if name.endswith(".csv"))

import pandas as pd
df = pd.read_csv(DATASET)
print(df["split"].value_counts())

In [ ]:
LABELS = ["legitimate", "scam", "spam"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}
SEED = 42

def part(split):
    d = df[df["split"] == split][["clean_message", "label", "language"]].reset_index(drop=True)
    return d.rename(columns={"clean_message": "text"})

train, val, synth_test = part("synth_train"), part("synth_val"), part("synth_test")
real, reconstructed = part("real_eval"), part("real_reconstructed")
assert not set(real["text"]) & set(train["text"]), "a real message leaked into training"
print(len(train), "train |", len(val), "val |", len(synth_test), "synthetic test |", len(real), "real |", len(reconstructed), "reconstructed")

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_dataset(frame):
    ds = Dataset.from_pandas(frame.assign(labels=frame["label"].map(label2id))[["text", "labels"]])
    # SMS are short: 128 tokens leaves plenty of room.
    return ds.map(lambda b: tokenizer(b["text"], truncation=True, max_length=128), batched=True)

train_ds, val_ds = to_dataset(train), to_dataset(val)

In [ ]:
import numpy as np
from sklearn.metrics import f1_score
from transformers import (AutoModelForSequenceClassification, DataCollatorWithPadding,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id)

def macro_f1(eval_pred):
    logits, labels = eval_pred
    return {"macro_f1": f1_score(labels, np.argmax(logits, axis=-1), average="macro")}

args = TrainingArguments(
    output_dir="mbert_scamshield",
    num_train_epochs=4,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    save_total_limit=1,
    seed=SEED,
    fp16=torch.cuda.is_available(),
    report_to="none",
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
    data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=macro_f1,
    # The training set is small, so stop once validation macro-F1 stops improving.
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)
trainer.train()

In [ ]:
# Same scoring as ml/src/metrics.py, so the numbers compare directly with the TF-IDF models.
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score, classification_report

def scam_metrics(y_true, y_pred, y_score=None):
    y_true, y_pred = np.asarray(y_true, bool), np.asarray(y_pred, bool)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=True, zero_division=0)
    out = {"n": int(len(y_true)), "n_scam": int(y_true.sum()), "accuracy": float(accuracy_score(y_true, y_pred)),
           "precision": float(p), "recall": float(r), "f1": float(f1)}
    if y_score is not None and len(np.unique(y_true)) == 2:
        out["auc_roc"] = float(roc_auc_score(y_true, y_score))
    return out

def bootstrap_ci(y_true, y_pred, y_score=None, n_boot=1000, seed=42):
    y_true, y_pred = np.asarray(y_true, bool), np.asarray(y_pred, bool)
    y_score = None if y_score is None else np.asarray(y_score)
    rng, samples = np.random.default_rng(seed), {}
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_true), len(y_true))
        if len(np.unique(y_true[idx])) < 2:
            continue
        m = scam_metrics(y_true[idx], y_pred[idx], None if y_score is None else y_score[idx])
        for k, v in m.items():
            if k not in ("n", "n_scam"):
                samples.setdefault(k, []).append(v)
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in samples.items()}

def predict(frame):
    logits = trainer.predict(to_dataset(frame)).predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    return np.array([id2label[i] for i in probs.argmax(-1)]), probs[:, label2id["scam"]]

In [ ]:
results = {"model": "mbert", "base_model": MODEL_NAME,
           "versions": {"transformers": transformers.__version__, "torch": torch.__version__}}

# Headline: real messages, scam vs not-scam (spam and legitimate both count as not-scam).
pred, scam_prob = predict(real)
y_true = real["label"].eq("scam").to_numpy()
m = scam_metrics(y_true, pred == "scam", scam_prob)
m["ci95"] = bootstrap_ci(y_true, pred == "scam", scam_prob)
m["real_label_vs_predicted"] = pd.crosstab(real["label"], pd.Series(pred, name="predicted")).reindex(columns=LABELS, fill_value=0).to_dict(orient="index")
results["A_synthetic_to_real_scam_vs_not"] = m

# Synthetic test, all three labels.
pred_s, _ = predict(synth_test)
report = classification_report(synth_test["label"], pred_s, labels=LABELS, output_dict=True, zero_division=0)
results["C_synthetic_test_3way"] = {"macro_f1": report["macro avg"]["f1-score"], "accuracy": report["accuracy"],
                                    "per_class": {l: report[l] for l in LABELS}}

if len(reconstructed):
    pred_r, _ = predict(reconstructed)
    results["reconstructed_scam_detection_rate"] = float(np.mean(pred_r == "scam"))

ci = m["ci95"]["f1"]
print(f"Real messages: F1 {m['f1']:.3f} (95% range {ci[0]:.2f} to {ci[1]:.2f}), precision {m['precision']:.3f}, "
      f"recall {m['recall']:.3f}, AUC {m.get('auc_roc', float('nan')):.3f}")
print(f"Synthetic test: macro-F1 {results['C_synthetic_test_3way']['macro_f1']:.3f}")

In [ ]:
# Download the results (numbers only, no message text) and put the file in ml/models/ on your PC.
import json, os
with open("mbert_results.json", "w") as f:
    json.dump(results, f, indent=2)
files.download("mbert_results.json")

# Remove the private dataset from this Colab session.
os.remove(DATASET)
print("Deleted", DATASET, "from the Colab session.")

## After the run

1. Put `mbert_results.json` in `ml/models/` and commit it; it holds numbers only.
2. Compare its `A_synthetic_to_real_scam_vs_not` F1 and range with the TF-IDF rows in
   `ml/data/DATASET_README.md`. With only 113 real messages, overlapping ranges mean
   the models cannot be told apart, so say so rather than calling one the winner.
3. Note in the report which transformers version you used (printed in the first cell) and
   that training stopped early if it did (see the training log above).